# P20 — Mamba: modelado de secuencias en tiempo lineal con espacios de estados selectivos

## 1. Título y paper

**Paper:** *Mamba: Linear-Time Sequence Modeling with Selective State Spaces*  
**Autoría:** Albert Gu, Tri Dao  
**Año y venue:** 2023 · arXiv:2312.00752 · COLM 2024 (Outstanding Paper Award)  
**Nivel:** L4 · **Motor:** `ssm`  
**Ficha completa:** [`P20_mamba`](../../papers/foundational/P20_mamba/README.md)

**Hito:** El primer competidor serio del Transformer en lenguaje: tiempo lineal y estado de tamaño fijo, sin atención.

- [arXiv:2312.00752](https://arxiv.org/abs/2312.00752)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: La atención cuesta O(n²) y su memoria crece con la secuencia; las alternativas subcuadráticas previas no alcanzaban a la atención en lenguaje.
2. Ejecutar una implementación mínima de la propuesta: Hacer que los parámetros del espacio de estados dependan de la ENTRADA (selección), y compensar la pérdida de la convolución eficiente con un algoritmo paralelo consciente del hardware.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P03
- P08
- S4 y los espacios de estados estructurados


## 4. Intuición

Un RNN comprime el pasado en un estado fijo —barato pero olvida—; la atención guarda todo —recuerda pero cuesta n²—. Mamba se queda con el estado fijo y añade lo que le faltaba: **la puerta decide según lo que está leyendo**, no según una regla fija.


## 5. Concepto mínimo

```text
SSM invariante en el tiempo (S4 y anteriores):
    h_t = A·h_{t−1} + B·x_t          A, B FIJAS  →  se puede convertir en convolución

SSM selectivo (Mamba):
    h_t = A(x_t)·h_{t−1} + B(x_t)·x_t    A, B DEPENDEN DE LA ENTRADA
```

Ese cambio rompe la convolución eficiente —por eso hace falta un escaneo paralelo consciente del hardware— pero es lo que permite **razonar sobre el contenido**: ignorar relleno y retener lo relevante.


## 6. Código explicado

El motor compara ambos en una tarea de copia selectiva: recordar tokens marcados entre relleno.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('ssm', seed=7)['result']
print('tokens:', r['tokens'], '· marcados:', r['marcados'], '\n')
show(r['invariante_en_el_tiempo'])
show(r['selectivo'])
print('mejora de separacion:', r['mejora_de_separacion'])

## 7. Predicción antes de ejecutar

1. ¿Podrá el SSM invariante distinguir los tokens marcados del relleno?
2. ¿Qué le pasa a la memoria de la atención cuando n pasa de 1 000 a 100 000?
3. ¿Y a la del SSM?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for fila in r['complejidad']:
    print(f"n={fila['n']:>7} · attn {fila['attention_ops']:>15,} ops / KV {fila['attention_memoria_kv']:>10,} "
          f"· ssm {fila['ssm_ops']:>12,} ops / estado {fila['ssm_memoria_estado']:>6,}")

## 9. Salida interpretable

La memoria del SSM es **constante**: `d·N` no depende de la longitud. La de la atención crece linealmente con la secuencia (la caché KV) y su cómputo, cuadráticamente. Ese es el argumento económico; el argumento de calidad es la separación que acabas de medir.


## 10. Comentario pedagógico

El compromiso es real y conviene enunciarlo sin entusiasmo: un estado de tamaño fijo **es** una compresión con pérdida. La atención puede volver a mirar cualquier token exacto; el SSM solo tiene lo que decidió guardar. Por eso proliferaron los híbridos que alternan ambos bloques.


## 11. Error o anti-patrón deliberado

Anti-patrón: «Mamba sustituye al Transformer». Es una afirmación de arquitectura sin tarea, dato ni escala.


In [ ]:
print('«Mamba sustituye al Transformer» ← ¿en qué tarea, a qué escala, con qué presupuesto?')
print('El paper reporta resultados hasta cierto tamaño y en ciertas modalidades.')
print('Extrapolar de ahi a «sustituye» es narrativa, no evidencia.')

## 12. Corrección

Enunciado defendible, con sus condiciones:


In [ ]:
defendible = {
    'coste': 'tiempo lineal en la longitud y estado de memoria constante',
    'calidad': 'competitivo con Transformers de tamaño comparable en las tareas del paper',
    'mecanismo': 'la seleccion dependiente de la entrada es lo que da razonamiento sobre contenido',
    'coste_oculto': 'un estado fijo es compresion con perdida: no hay recuperacion exacta',
    'no_demostrado': 'paridad general con la atencion a cualquier escala y tarea',
}
show(defendible)

## 13. Desafío guiado

Sube la proporción de tokens marcados y observa cuándo la selección deja de ayudar.


In [ ]:
for cada in (17, 7, 3, 2):
    marcados = len([i for i in range(60) if i % cada == 3])
    print(f'1 de cada {cada:>2} tokens marcado → {marcados:>2}/60 relevantes '
          f"({'seleccionar aporta poco' if marcados > 20 else 'seleccionar aporta mucho'})")

## 14. Desafío autónomo

Implementa la tarea de copia selectiva del paper con longitudes crecientes y entrena dos modelos de tamaño comparable: uno con puertas fijas y otro con puertas dependientes de la entrada. Reporta exactitud frente a longitud y memoria máxima usada.


## 15. Evidencia de aprendizaje

Guarda la separación de ambos modelos, la tabla de complejidad y tu enunciado defendible sobre qué gana y qué pierde frente a la atención.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P20_mamba/README.md) · evaluación formal: [`assessments/papers/P20_mamba.md`](../../assessments/papers/P20_mamba.md)


## 16. Cierre

Se puede abaratar el **eje de la secuencia**. Queda el otro eje: el de los parámetros, donde cada token paga por todos aunque no los necesite.


## 17. Conexión con el siguiente hito

- arquitecturas híbridas atención + SSM

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
